# CCNY Fall 2021 Calendar Scraper

Scrape the academic calendar using requests, Beautiful Soup, and pandas.

In [2]:
%pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

url = "https://www.ccny.cuny.edu/registrar/fall"
fallback = Path("data/fall_2021.html")

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    html = response.text
    source = "Live CCNY page (requests)"
except requests.RequestException as error:
    if not fallback.exists():
        raise RuntimeError("Download failed and data/fall_2021.html is missing.") from error
    html = fallback.read_text(encoding="utf-8")
    source = "Saved browser calendar table (live request failed)"
    print(f"Live download unavailable: {type(error).__name__}")

soup = BeautifulSoup(html, "html.parser")
if not soup.find("table") or "Fall 2021 Academic Calendar" not in soup.get_text():
    raise ValueError("The response does not contain the expected calendar.")

print("Source:", source)
print(soup.title.get_text(strip=True))


Live download unavailable: HTTPError
Source: Saved browser calendar table (live request failed)
Fall 2021 Academic Calendar


### Download issue

I got a 403 Forbidden error when I tried downloading the calendar with requests. Adding a User-Agent header did not fix it. The notebook tries the website first, then uses a local copy of the calendar table if the download fails. This lets me continue parsing the calendar with Beautiful Soup, but the live download with requests is still blocked.

### Extract the calendar rows

I use Beautiful Soup to get the date, day, and description from each table row. The header uses th cells, so it is skipped.

In [12]:
table = soup.find("table")
rows = []

for row in table.find_all("tr"):
    cells = row.find_all("td")

    if len(cells) == 3:
        values = [
            " ".join(cell.get_text(" ", strip=True).split())
            for cell in cells
        ]
        rows.append(values)

print("Calendar entries:", len(rows))
rows[:3]


Calendar entries: 36


[['August 01',
  'Sunday',
  'Application for degree for January and February 2022 begins'],
 ['August 18', 'Wednesday', 'Last day to apply for Study Abroad'],
 ['August 24',
  'Tuesday',
  'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;']]

### Create a DataFrame

I put the extracted rows into a pandas DataFrame. The columns are date_text, dow (day of the week), and text (the description). For now, the dates are strings. I will convert them to Python dates in the next step.

In [13]:
raw_df = pd.DataFrame(rows, columns=["date_text", "dow", "text"])
raw_df.head()


,date_text,dow,text
0,August 01,Sunday,Application for degree for January and Februar...
1,August 18,Wednesday,Last day to apply for Study Abroad
2,August 24,Tuesday,Last day of Registration; Last day to file ePe...
3,August 25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
4,August 25 - 31,Wednesday - Tuesday,Change of program period; late fees apply


### Convert dates and set the index

I expand each date range into one row per day and repeat its description. I use 2021 for dates without a year and keep the explicit 2022 date. I calculate dow from each date. Dates can appear more than once when different events happen on the same day.


In [14]:
from datetime import datetime, timedelta, date

def expand_dates(value):
    value = value.replace("–", "-").replace("—", "-")

    if "-" in value:
        start_text, end_day = value.split("-", maxsplit=1)
        start = datetime.strptime(start_text.strip() + " 2021", "%B %d %Y").date()
        end = start.replace(day=int(end_day.strip()))
        return [
            start + timedelta(days=offset)
            for offset in range((end - start).days + 1)
        ]

    if "," in value:
        return [datetime.strptime(value, "%B %d, %Y").date()]

    return [datetime.strptime(value + " 2021", "%B %d %Y").date()]

records = []
for row in raw_df.itertuples(index=False):
    for calendar_date in expand_dates(row.date_text):
        records.append({
            "date": calendar_date,
            "dow": calendar_date.strftime("%A"),
            "text": row.text,
        })

df = pd.DataFrame(records).set_index("date").sort_index(kind="stable")
df


,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-27,Friday,Change of program period; late fees apply
2021-08-28,Saturday,Change of program period; late fees apply


### Check the result

The index should contain Python date objects, and the two columns should be dow and text.


In [15]:
assert all(type(value) is date for value in df.index)
assert df.columns.tolist() == ["dow", "text"]
assert df.index.is_monotonic_increasing
assert df["text"].str.strip().ne("").all()
assert date(2022, 1, 1) in df.index

print("Calendar rows:", len(df))
print("Index value type:", type(df.index[0]))


Calendar rows: 58
Index value type: <class 'datetime.date'>
